# DataCrunch 2 — Offline Transformer Pretraining

**Run this on your own GPU / supercomputer — this notebook is NOT submitted
to CrunchDAO.** Its only job is to produce `transformer_checkpoint.pt`,
which you then upload as a resource alongside the separate submission
notebook (`competition_transformer.ipynb`).

Why split it this way: CrunchDAO's own runtime is CPU-only (16 vCPU, no
GPU, per your run details) and calls `train()` 9 times inside a shared
~10h/week compute quota. Full transformer training from scratch doesn't
fit that budget. Pretraining here — with real compute and no time
pressure — then having each CrunchDAO-side `train()` call do only a small,
cheap fine-tune on newly available data is how the two constraints get
reconciled. This split-training pattern is explicitly supported by
CrunchDAO: model files can be uploaded as resources alongside a notebook
submission, and their own FAQ describes training locally and having the
platform continue training with more data during the Out-of-Sample phase.

**⚠️ Keep the `FTTransformerRegressor` class definition below byte-for-byte
identical to the copy in the submission notebook.** The checkpoint is a
`state_dict` — it only loads into a model with the exact same architecture
that produced it.


In [1]:
# Adjust for your CUDA version if needed -- see https://pytorch.org/get-started/locally/
%pip install crunch-cli torch --upgrade --quiet --progress-bar off

# Pulls the same competition data used by the submission notebook
!crunch setup-notebook datacrunch-2 0JiCmmP21Ca88X8TApRuDHMH --size small

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.24.0 requires torch==2.9.0, but you have torch 2.14.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
zsh:1: /Users/Tia_987/Downloads/datacrunch/.venv/bin/crunch: bad interpreter: /Users/Tia_987/Downloads/AML25-main/.venv/bin/python3: no such file or directory


In [2]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr

import crunch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- pretraining here will be slow. Check your CUDA setup.")

crunch_tools = crunch.load_notebook()

Using device: cpu
loaded crunch tools for module: <module '__main__'>

cli version: 12.0.2
available ram: 16.00 gb
available cpu: 10 core
----


/Users/Tia_987/Downloads/datacrunch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config & helpers

In [3]:
RANDOM_STATE = 0
ID_COLUMNS = ["id", "moon"]
CHECKPOINT_PATH = "transformer_checkpoint.pt"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def get_feature_columns(df: pd.DataFrame):
    return [c for c in df.columns if c not in ID_COLUMNS and c != "target"]


def spearman(y_true, y_pred) -> float:
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(corr) else corr

## Model

Feature-Tokenizer Transformer (Gorishniy et al. 2021, adapted): each of the
already-quantized (0–6) features becomes a token via a value embedding plus
a learned feature-identity embedding — identity has to be injected
explicitly since a plain transformer is permutation-invariant over its
input tokens, the same role positional embeddings play for text. A learned
`[CLS]` token is prepended; its output after the encoder feeds a small
regression head.

In [4]:
class FTTransformerRegressor(nn.Module):
    def __init__(self, n_features: int, n_bins: int = 7, d_model: int = 64,
                 n_heads: int = 4, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.n_features = n_features
        self.value_embedding = nn.Embedding(n_bins, d_model)
        self.feature_id_embedding = nn.Embedding(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_bins: torch.Tensor) -> torch.Tensor:
        # x_bins: (batch, n_features) int64 in [0, n_bins)
        batch_size = x_bins.shape[0]
        feature_ids = torch.arange(self.n_features, device=x_bins.device).unsqueeze(0)
        tokens = self.value_embedding(x_bins) + self.feature_id_embedding(feature_ids)
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = encoded[:, 0, :]
        return self.head(cls_out).squeeze(-1)


def build_model(config: dict) -> FTTransformerRegressor:
    return FTTransformerRegressor(
        n_features=config["n_features"],
        n_bins=config.get("n_bins", 7),
        d_model=config.get("d_model", 64),
        n_heads=config.get("n_heads", 4),
        n_layers=config.get("n_layers", 3),
        dropout=config.get("dropout", 0.1),
    )


def save_checkpoint(path, model, config, feature_columns, target_mean, target_std):
    torch.save({
        "model_state": model.state_dict(),
        "config": config,
        "feature_columns": feature_columns,
        "target_mean": target_mean,
        "target_std": target_std,
    }, path)


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = build_model(checkpoint["config"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    return model, checkpoint

## Load data

In [5]:
X_train, y_train, X_test = crunch_tools.load_data()
feature_columns = get_feature_columns(X_train)
print(f"{len(feature_columns)} features, {len(X_train):,} rows")

# Features are already quantized into 7 bins (0-6) -- int8 instead of the
# default float64 cuts memory roughly 8x. This is the single biggest lever
# on the ~40GB RAM footprint you mentioned.
X_bins_all = X_train[feature_columns].to_numpy().astype(np.int8)
print(f"Feature matrix memory: {X_bins_all.nbytes / 1e9:.2f} GB (int8)")

data/X.reduced.small.zip: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/X.reduced.small.zip (100411670 bytes)
data/X.reduced.small.zip: already exists, file length match
data/X.reduced.small.zip: already uncompressed, marker is present
data/moons_split.json: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/moons_split.json (11088 bytes)
data/moons_split.json: already exists, file length match
data/y.reduced.small.zip: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/y.reduced.small.zip (1158625 bytes)
data/y.reduced.small.zip: already exists, file length match
data/y.reduced.small.zip: already uncompressed, marker is present
1150 features, 226,455 rows
Feature matrix memory: 0.26 GB (int8)


## Held-out split & target scaling

Standardizing the target helps training stability (MSE loss on raw skewed
returns can be noisy). We don't need to invert this at inference — Spearman
rank correlation is invariant to any monotonic affine transform, so a
standardized prediction preserves the same ranking as an unstandardized
one. `target_mean`/`target_std` still get saved in the checkpoint so the
CrunchDAO-side fine-tuning uses the exact same scale, rather than each
call computing its own and drifting.

In [6]:
moons = np.sort(X_train["moon"].unique())
holdout_moons = moons[-50:]
is_holdout = X_train["moon"].isin(holdout_moons).to_numpy()

target = y_train["target"].to_numpy()
target_mean, target_std = float(target[~is_holdout].mean()), float(target[~is_holdout].std())
target_scaled = (target - target_mean) / target_std

X_fit_bins, y_fit = X_bins_all[~is_holdout], target_scaled[~is_holdout]
X_val_bins, y_val_scaled = X_bins_all[is_holdout], target_scaled[is_holdout]
y_val_raw = target[is_holdout]

print(f"train rows: {len(y_fit):,}   holdout rows: {len(y_val_scaled):,}")

train rows: 130,903   holdout rows: 95,552


## Pretrain

This is the notebook you have real compute for — feel free to sweep
`PRETRAIN_CONFIG` / epochs / learning rate here. Tracks best-epoch Spearman
on the held-out moons and keeps those weights (not necessarily the
last epoch's) for the checkpoint.

In [7]:
PRETRAIN_CONFIG = {
    "n_bins": 7,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 3,
    "dropout": 0.1,
}
PRETRAIN_EPOCHS = 30
PRETRAIN_LR = 3e-4
PRETRAIN_BATCH_SIZE = 8192
WEIGHT_DECAY = 1e-5

In [ ]:
class BinDataset(torch.utils.data.Dataset):
    def __init__(self, X_bins, y):
        # self.X_bins = torch.from_numpy(X_bins).long()
        # self.y = torch.from_numpy(y).float()
        self.X_bins = self.X_bins
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_bins[idx], dtype=torch.long), 
            torch.tensor(self.y[idx], dtype=torch.float32)
        )


train_loader = torch.utils.data.DataLoader(
    BinDataset(X_fit_bins, y_fit),
    batch_size=PRETRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=(DEVICE == "cuda"),
)

del X_train
del y_train

model = build_model({**PRETRAIN_CONFIG, "n_features": len(feature_columns)}).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)
loss_fn = nn.MSELoss()

X_val_tensor = torch.from_numpy(X_val_bins).long().to(DEVICE)

best_val_spearman = -1.0
best_state = None

for epoch in range(PRETRAIN_EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    scheduler.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_tensor).cpu().numpy()
    val_spearman = spearman(y_val_raw, val_pred)
    print(f"epoch {epoch + 1}/{PRETRAIN_EPOCHS}  train_loss={total_loss / len(y_fit):.4f}  val_spearman={val_spearman:.4f}")

    if val_spearman > best_val_spearman:
        best_val_spearman = val_spearman
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"\nBest held-out Spearman during pretraining: {best_val_spearman:.4f}")

Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=95, pipe_handle=131)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'BinDataset'
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=95, pipe_handle=110)
                                            

RuntimeError: DataLoader worker (pid(s) 25906) exited unexpectedly

## Save checkpoint

Upload this file as a resource alongside your CrunchDAO notebook
submission (per CrunchDAO's docs: model files uploaded with a notebook
submission land in the `resources/` directory, the same directory passed
to `train()`/`infer()` as `model_directory_path`). After submitting, check
the run log for a line confirming the file was found — similar to how
`model.joblib` showed up in your earlier successful sklearn-based run —
so you know it warm-started rather than silently cold-starting on CPU.

In [ ]:
save_checkpoint(
    CHECKPOINT_PATH, model,
    config={**PRETRAIN_CONFIG, "n_features": len(feature_columns)},
    feature_columns=feature_columns,
    target_mean=target_mean, target_std=target_std,
)
print(f"Saved checkpoint to {CHECKPOINT_PATH}")
print("Upload this file as a resource alongside competition_transformer.ipynb.")